maturin + mm0-rs
just do python version from scratch
import vs export

my datalog provenance post https://www.philipzucker.com/metamath-datalog-provenenance/

https://digama0.github.io/mm0/thesis.pdf thesis

https://github.com/garrettkatz/mmpy
https://github.com/david-a-wheeler/mmverify.py

metamath hare?
https://github.com/gleachkr/mm0-zig

sudo apt-get install metamath

https://github.com/digama0/mm0/blob/master/mm0.md

https://grahamlk.me/Aufbau/#hilbert

Hmm. The variable condition is goofy enough, I wonder if a regular egraph could handle it.

rpython? it kind of is a bytecode interpreter. Hmm. Not really loops though, so like would the jit ever kick in? Prob not. You like run through in a single pass.


@relation
@congruence



In [2]:
%%file /tmp/hilbert.mm0
delimiter $ ( ) $;
provable sort wff;
term imp (a b: wff): wff; infixr imp: $->$ prec 25;
term not (a: wff): wff; prefix not: $not$ prec 35;
axiom h1 (a b: wff): $ a -> (b -> a) $;
axiom h2 (a b c: wff):
  $ (a -> (b -> c)) -> ((a -> b) -> (a -> c)) $;
axiom h3 (a b: wff): $ (not a -> not b) -> (b -> a) $;
axiom con2 (a b: wff): $ (a -> not b) -> (b -> not a) $;
axiom inot (a: wff): $ (a -> not a) -> not a $;
axiom mp (a b: wff): $ a $ > $ a -> b $ > $ b $;
theorem imp_refl (a: wff): $ a -> a $;
theorem hs (a b c: wff): $ a -> b $ > $ b -> c $ > $ a -> c $;
theorem notnot1 (p: wff): $ p -> not not p $;
theorem dne (p: wff): $ not not p -> p $;


Overwriting /tmp/hilbert.mm0


In [86]:
from dataclasses import dataclass,field
@dataclass(frozen=True)
class Sort:
    name : str
    provable : bool = False

    def __repr__(self):
        return self.name

wff = Sort("wff", provable=True)

@dataclass(frozen=True)
class Decl:
    name : str
    args : list[Sort]
    out : Sort
    #bound_args : list[Sort] = field(default_factory=list)
    #args : list[Sort] = field(default_factory=list)# name,sort tuples?

    def __call__(self, *args):
        # type check
        if len(args) != len(self.args):
            raise TypeError(f"{self.name} takes {len(self.args)} arguments, got {len(args)}")
        for i, (arg, sort) in enumerate(zip(args, self.args)):
            if arg.sort != sort:
                raise TypeError(f"Argument {i} of {self.name} should be of sort {sort.name}, got {arg.sort.name}")
        return App(self, list(args)) 
    

imp = Decl("imp", [wff, wff], wff)
not_ = Decl("not", [wff], wff)

def Consts(names, sort):
    return [Decl(name, [], sort)() for name in names.split()]


# App is a judgement
@dataclass(frozen=True)
class App:
    decl : Decl
    args : list["Term"]

    def __repr__(self):
        if len(self.args) == 0:
            return self.decl.name
        return f"{self.decl.name}({', '.join([str(arg) for arg in self.args])})"
    @property
    def sort(self):
        return self.decl.out

    @property
    def vars(self):
        return set.union(*[arg.vars for arg in self.args])


@dataclass(frozen=True)
class Var:
    name : str
    sort : Sort

    def __repr__(self):
        return self.name

    @property
    def vars(self):
        return set([self])
    
def Vars(names, sort):
    return [Var(name, sort) for name in names.split()]

@dataclass(frozen=True)
class BVar:
    name : str
    sort : Sort

    @property
    def vars(self):
        return set([self])


type Term = App | Var | BVar

def subst(t : Term, substs : dict[Var, Term]) -> Term:
    if isinstance(t, Var):
        return substs.get(t, t) # fail?
    elif isinstance(t, App):
        return App(t.decl, [subst(arg, substs) for arg in t.args])
    elif isinstance(t, BVar):
        return t
    else:
        raise TypeError(f"Unknown term type: {type(t)}")

from typing import Optional

@dataclass(frozen=True)
class ProofDecl:
    name : str
    vars : list[Sort]
    hyps : list[Term]
    concl : Term
    pf : Optional["ProofApp"]
    def __post_init__(self):
        assert self.concl.sort.provable, "conclusion must be provable"
        assert all(h.sort.provable for h in self.hyps), "hypotheses must be provable"
        assert all(isinstance(v, Var) for v in self.vars), "vars must be variables"
        assert len(set(self.vars)) == len(self.vars), "vars must be distinct"
        # do I need this one?
        assert self.concl.vars.union(*[hyp.vars for hyp in self.hyps]) <= set(self.vars), "all variables in conclusion and hypotheses must be declared"
        if self.pf is not None:
            assert self.pf.concl == self.concl, "proof conclusion does not match declared conclusion"
    def __repr__(self):
        return f"{"axiom" if self.pf is None else "theorem"} {self.name} ({', '.join([f'{v.name}: {v.sort}' for v in self.vars])}): {' > '.join([str(h) for h in self.hyps] + [str(self.concl)])}"
    
    def __call__(self, *args):
        assert len(args) >= len(self.vars), "not enough arguments to " + self.name
        assert all(v.sort == a.sort for a,v in zip(args, self.vars)), "argument sort mismatch"
        assert len(args) == len(self.vars) + len(self.hyps), "wrong number of arguments to " + self.name
        subs = {v: a for v,a in zip(self.vars, args)}
        for n, (hyp,a) in enumerate(zip(self.hyps, args[len(self.vars):])):
            if subst(hyp, subs) != a.concl:
                raise TypeError(f"Required Hypothesis {n}: {subst(hyp,subs)} does not match argument {a}")
        return ProofApp(self, list(args[:len(self.vars)]), list(args[len(self.vars):]))

def axiom(name, vars, hyps, concl):
    return ProofDecl(name, vars, hyps, concl, None)

def prove(name, vars, hyps, concl, pf):
    assert pf.concl == concl
    for hyp in pf.hyps:
        assert hyps[hyp.pos] == hyp.concl
    return ProofDecl(name, vars, hyps, concl, pf)


a,b,c = Vars("a b c", wff) #Var("a", wff), Var("b", wff)
#axiom h1 (a b: wff): $ a -> (b -> a) $;
#p,q,r = Decl("p", [], wff)(), Decl("q", [], wff)(), Decl("r", [], wff)()
p,q,r = Consts("p q r", wff)



@dataclass(frozen=True)
class ProofApp:
    decl : ProofDecl
    args : list[Term]
    subpfs : list["ProofApp"]

    @property
    def concl(self):
        return subst(self.decl.concl, {v: a for v,a in zip(self.decl.vars, self.args)})
    
    def __repr__(self):
        return f"{list(sorted(self.hyps))} |- {self.concl}"
    
    @property
    def hyps(self) -> set["Hyp"]:
        return set().union(*[hyp.hyps for hyp in self.subpfs])

@dataclass(frozen=True)
class Hyp:
    pos : int
    concl : Term

    @property
    def hyps(self) -> set["Hyp"]:
        print(type(self))
        return {self}
    
    def __repr__(self):
        return self.concl
    




#imp_refl = prove("imp_refl", [a], [], imp(a,a), mp(a, imp(a,a), h1(a, a)))

"""
imp_refl
--------
l1: $ a -> ((a -> a) -> a) $ by h1 
l2: $ a -> (a -> a) $ by h1 
l3: $ (a -> ((a -> a) -> a)) -> ((a -> (a -> a)) -> (a -> a)) $ by h2 
l4: $ (a -> (a -> a)) -> (a -> a) $ by mp [l1, l3]
l5: $ a -> a $ by mp [l2, l4]
"""
mp = axiom("mp", [a,b], [a, imp(a,b)], b)
h1 = axiom("h1", [a,b], [], imp(a, imp(b, a)))
h2 = axiom("h2", [a,b,c], [], imp(imp(a, imp(b, c)), imp(imp(a, b), imp(a, c))))
l1 = h1(a, imp(a,a))
l2 = h1(a, a)
l3 = h2(a,imp(a,a),a)
l4 = mp(l1.concl, l3.concl.args[1], l1, l3)
l5 = mp(l2.concl, l4.concl.args[1], l2, l4)
imp_refl = prove("imp_refl", [a], [], imp(a,a), l5)
imp_refl

l1
l2

[] |- imp(a, imp(a, a))

In [87]:
"""
theorem hs (a b c: wff): $ a -> b $ > $ b -> c $ > $ a -> c $;
hs
---
l1: $ (b -> c) -> (a -> (b -> c)) $ by h1 
l2: $ a -> (b -> c) $ by mp [#2, l1]
l3: $ (a -> (b -> c)) -> ((a -> b) -> (a -> c)) $ by h2 
l4: $ (a -> b) -> (a -> c) $ by mp [l2, l3]
l5: $ a -> c $ by mp [#1, l4]
"""
_0 = Hyp(0, imp(a,b))
_1 = Hyp(1, imp(b,c))
l1 = h1(imp(b, c), a)
l2 = mp(_1.concl, l1.concl.args[1], _1, l1)
l3 = h2(a, b, c)
l4 = mp(l2.concl, l3.concl.args[1], l2, l3)
l5 = mp(_0.concl, l4.concl.args[1], _0, l4)
#hs = prove("hs", [a,b,c], [imp(a,b), imp(b,c)], imp(a,c), l5)
l5.hyps

<class '__main__.Hyp'>


TypeError: unhashable type: 'list'

In [ ]:
@dataclass
class Assertion:
    name : str
    bvars : list[Decl]
    vars : list[Decl]
    hyps : list[Term]
    body : Term
    proof : ProofTerm | None
    #proof : AppAssertion #list[tuple[Term, tuple[]]]

    # check?
    def __call__(self, *args):
        return PfApp(self, args)
    
    def check(self):
        if self.proof is None:
            return True
        else:
            

        pass
    def __post_init__(self):
        assert check


@dataclass
class PfApp:
    assertion : Assertion
    args : list[Term]
    assertargs : list[ProofTerm] 

    def check(self, ctx):
        # check is valid proof of assertion

    def conc(self):
        subst(self.assertion.body, {var.name: arg for var, arg in zip(self.assertion.vars, self.args)})


type ProofTerm = PfApp | int

In [ ]:



"""
@dataclass
class AppAssertion:
    ctx: list[Term]
    assertion : Proof | int
    args : list[Term]
    assertargs : list[AppAssertions] #?
"""


@dataclass
class Axiom:
    pass
    #assertion : Assertion

@dataclass
class ProofTerm:
    assertion : Assertion

type Proof = ProofTerm | Axiom

#@dataclass
#class Proof

def axiom(name, vars, bvars=[]):
    return Assertion(name, vars, bvars, proof=None)

def theorem(name, vars, proof, bvars=[]):
    # check proof?
    return Assertion(name, vars, bvars, proof=proof)

def subst(ctx : list[Var], a : Assertion, *subst):
    # 
    

If defns are just unfolded, maybe I could just use python meta functions as defns

I could implement this over the z3 ast with some meta carrying around what is bvar
But I dunno if the bvar notion would play right?


